In [1]:
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 9.6 MB/s eta 0:00:00


In [2]:
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.naive import NaiveForecaster
from sktime.datasets import load_airline
from sktime.split import temporal_train_test_split
import numpy as np
y = load_airline()
y_train, y_test = temporal_train_test_split(y, test_size=6)

In [3]:
y_train

,Number of airline passengers
Period,
1949-01,112.0
1949-02,118.0
1949-03,132.0
1949-04,129.0
1949-05,121.0
...,...
1960-02,391.0
1960-03,419.0
1960-04,461.0


In [4]:
y_test

,Number of airline passengers
Period,
1960-07,622.0
1960-08,606.0
1960-09,508.0
1960-10,461.0
1960-11,390.0
1960-12,432.0


In [5]:
ForecastingHorizon([1, 2, 3])
# ForecastingHorizon([1, 2, 3], is_relative=True)

ForecastingHorizon([1, 2, 3], dtype='int64', is_relative=True)

In [6]:
ForecastingHorizon(np.arange(1, 7))
# ForecastingHorizon([1, 2, 3, 4, 5, 6], is_relative=True)

ForecastingHorizon([1, 2, 3, 4, 5, 6], dtype='int64', is_relative=True)

In [7]:
ForecastingHorizon(y_test.index, is_relative=False)
# ForecastingHorizon(['1960-07', ..., '1960-12'], is_relative=False)

ForecastingHorizon(['1960-07', '1960-08', '1960-09', '1960-10', '1960-11', '1960-12'], dtype='period[M]', name='Period', is_relative=False)

In [8]:
ForecastingHorizon(y_test, is_relative=False)
# ForecastingHorizon(['1960-07', ..., '1960-12'], is_relative=False)

TypeError: Invalid `fh`. The type of the passed `fh` values is not supported. Please use one of ('int', 'range', '1D np.ndarray of type int', '1D np.ndarray of type timedelta or dateoffset', 'list of type int', 'pd.RangeIndex', 'pd.PeriodIndex', 'pd.DatetimeIndex', 'pd.TimedeltaIndex'), but found type <class 'pandas.core.series.Series'>, values = Period
1960-07    622.0
1960-08    606.0
1960-09    508.0
1960-10    461.0
1960-11    390.0
1960-12    432.0
Freq: M, Name: Number of airline passengers, dtype: float64

In [ ]:
# The above error tells that only index is supported by ForecastingHorizon
#   - index can be specified via
#       1. int
#       2. list of type int
#       3. range -> np.arange which in turn returns a 1D np.ndarray of type int
#       4. 1D np.ndarray of type int
#       5. 1D np.ndarray of type timedelta or dateoffset
#       6. pd.RangeIndex
#       7. pd.PeriodIndex
#       8. pd.DatetimeIndex
#       9. pd.TimedeltaIndex

# Internally everything gets converted based on following logic:
#   - Single int: Converted to pd.Index([value], dtype=int)
#   - range object: Converted to pd.RangeIndex
#   - Single pd.Timedelta or date offset: Wrapped in pd.Index([value])
#   - np.ndarray (1D, int/timedelta): Converted to pd.Index
#   - list of int: Converted to pd.Index(values, dtype=int)
#   - Valid pandas Index types: RangeIndex, Int64Index, PeriodIndex, TimedeltaIndex, DatetimeIndex (no conversion needed)

# Validations performed
#   - Checks type compatibility and converts if necessary
#   - Validates no duplicates exist (raises ValueError)
#   - Sorts values before returning

# Relative vs. Absolute Index Type Handling

# The code distinguishes between two categories of pandas indices:

# Relative Index Types (distances from a reference point):

# pd.Index with integer values (e.g., [1, 2, 3])
# pd.RangeIndex
# pd.TimedeltaIndex (e.g., [Timedelta('1 days'), Timedelta('2 days')])
# Absolute Index Types (actual time points):

# pd.PeriodIndex (e.g., periods like '2021-01')
# pd.DatetimeIndex (e.g., timestamps like '2021-01-15')

# a general overview of ForecastingHorizon -> https://www.sktime.net/en/stable/examples/01_forecasting.html#Step-2---Specifying-the-forecasting-horizon

# ForecastingHorizon api reference: https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.base.ForecastingHorizon.html

# ForecastingHorizon will automatically assume a relative horizon
# if temporal difference types from pandas are passed;
# if value types from pandas are passed, it will assume an absolute horizon.


# Permissible parameters and their conversions

ForecastingHorizon can be specified via
1. int
2. list of type int
3. range -> np.arange which in turn returns a 1D np.ndarray of type int
4. 1D np.ndarray of type int
5. 1D np.ndarray of type timedelta or dateoffset
6. pd.RangeIndex
7. pd.PeriodIndex
8. pd.DatetimeIndex
9. pd.TimedeltaIndex

Internally everything gets converted based on following logic:
   - Single int: Converted to pd.Index([value], dtype=int)
   - range object: Converted to pd.RangeIndex
   - Single pd.Timedelta or date offset: Wrapped in pd.Index([value])
   - np.ndarray (1D, int/timedelta): Converted to pd.Index
   - list of int: Converted to pd.Index(values, dtype=int)
   - Valid pandas Index types: RangeIndex, Int64Index, PeriodIndex, TimedeltaIndex, DatetimeIndex (no conversion needed)

Validations performed
   - Checks type compatibility and converts if necessary
   - Validates no duplicates exist (raises ValueError)
   - Sorts values before returning
   - Index Contiguity Checking (`_is_contiguous`)

Index Type Conversion Methods: `to_pandas()` and `to_numpy()`

# Relative vs. Absolute Index Type Handling

Two categories of pandas indices:

- **Relative Index Types** (distances from a reference point):
  - pd.Index with integer values (e.g., [1, 2, 3])
  - pd.RangeIndex
  - pd.TimedeltaIndex (e.g., [Timedelta('1 days'), Timedelta('2 days')])

- **Absolute Index Types** (actual time points):
  - pd.PeriodIndex (e.g., periods like '2021-01')
  - pd.DatetimeIndex (e.g., timestamps like '2021-01-15')

ForecastingHorizon will automatically assume a relative horizon if temporal difference types from pandas are passed; but if value types from pandas are passed, it will assume an absolute horizon.
This automatic detection helps users avoid manual specification of horizon type.

Methods to perform conversions:
- `to_relative(cutoff)` converts absolute indices to relative offsets
- `to_absolute(cutoff)` converts relative offsets to absolute indices
- `_to_relative()` function cached with @lru_cache
- `_to_absolute()` function cached with @lru_cache

These are the most complex pandas-related operations:

to_relative(cutoff): Converts absolute indices to relative offsets

#  Frequency Management (`_check_freq`)

This is a pain point.

Frequency Extraction Logic:

- If obj is already a pd.offsets.BaseOffset (e.g., Day(), MonthEnd()), return as-is
- If obj has a .cutoff attribute (forecaster object), recursively extract from cutoff
- If obj is pd.Period, pd.Index, or string, extract/convert to offset via to_offset()
- Return None if frequency cannot be determined

The `freq.setter` property ensures frequency consistency throughout the horizon object's lifetime. If you try to set a conflicting frequency, it raises an error.


[A general overview of ForecastingHorizon](https://www.sktime.net/en/stable/examples/01_forecasting.html#Step-2---Specifying-the-forecasting-horizon)

[ForecastingHorizon api reference](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.base.ForecastingHorizon.html)



In [9]:
# set cutoff (last time point of training data)
cutoff = y_train.index[-1]
cutoff


Period('1960-06', 'M')

In [12]:
fh = ForecastingHorizon(y_test.index, is_relative=False)
fh

ForecastingHorizon(['1960-07', '1960-08', '1960-09', '1960-10', '1960-11', '1960-12'], dtype='period[M]', name='Period', is_relative=False)

In [13]:
# Converting to_relative
fh.to_relative(cutoff=cutoff)
# ForecastingHorizon([1, 2, 3, 4, 5, 6], is_relative=True)

ForecastingHorizon([1, 2, 3, 4, 5, 6], dtype='int64', is_relative=True)

In [17]:
fh = ForecastingHorizon([1, 2, 3, 4, 5, 6], is_relative=True)
fh

ForecastingHorizon([1, 2, 3, 4, 5, 6], dtype='int64', is_relative=True)

In [18]:
# Converting to_absolute
fh = fh.to_absolute(cutoff=cutoff)
fh
# ForecastingHorizon(['1960-07', ..., '1960-12'], is_relative=False)

ForecastingHorizon(['1960-07', '1960-08', '1960-09', '1960-10', '1960-11', '1960-12'], dtype='period[M]', is_relative=False)

In [19]:
# Automatically casted ForecastingHorizon from list when calling predict()
forecaster = NaiveForecaster(strategy="drift")
forecaster.fit(y_train)
y_pred = forecaster.predict(fh=[1,2,3])
forecaster.fh
# ForecastingHorizon([1, 2, 3], dtype='int64', is_relative=True)

ForecastingHorizon([1, 2, 3], dtype='int64', is_relative=True)

In [20]:
# This is identical to give an object of ForecastingHorizon
y_pred = forecaster.predict(fh=ForecastingHorizon([1,2,3]))
forecaster.fh
# ForecastingHorizon([1, 2, 3], dtype='int64', is_relative=True)

ForecastingHorizon([1, 2, 3], dtype='int64', is_relative=True)